In [ ]:
import logging
from datetime import datetime

import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset

from sentence_transformers import SentenceTransformer, LoggingHandler

In [ ]:
# Set the log level to INFO to get more information
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()])
output_path = "distilled/model-distillation-" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [ ]:
# Teacher Model: Model we want to distill to a smaller model
# teacher_model_name = "stsb-bert-base"
teacher_model_name = "sentence-transformers/stsb-roberta-base-v2"
teacher_model = SentenceTransformer(teacher_model_name)

In [ ]:
for i, layer_module in enumerate(teacher_model._first_module().auto_model.encoder.layer):
    print(i, layer_module)

In [ ]:
def deduplicate(dataset):
    df = pd.DataFrame(dataset)
    df = df.drop_duplicates()
    return Dataset.from_pandas(df, preserve_index=False)

# Concatenate all sentences into a new column "sentence"
def combine_sentences(batch):
    return {"sentence": batch["sentence1"] + batch["sentence2"]}

In [ ]:
logging.info("Load the AllNLI dataset")
# Load the AllNLI dataset: https://huggingface.co/datasets/sentence-transformers/all-nli
nli_train_dataset = load_dataset("sentence-transformers/all-nli", "pair-score", split="train")
nli_eval_dataset = load_dataset("sentence-transformers/all-nli", "pair-score", split="dev")
nli_train_dataset = nli_train_dataset.map(combine_sentences, batched=True, remove_columns=nli_train_dataset.column_names)
nli_eval_dataset = nli_eval_dataset.map(combine_sentences, batched=True, remove_columns=nli_eval_dataset.column_names)
nli_train_dataset = deduplicate(nli_train_dataset)
nli_eval_dataset = deduplicate(nli_eval_dataset)
logging.info(nli_train_dataset)

logging.info("Load the STSB dataset")
# Load the STSB dataset: https://huggingface.co/datasets/sentence-transformers/stsb
stsb_eval_dataset = load_dataset("sentence-transformers/stsb", split="validation")
stsb_test_dataset = load_dataset("sentence-transformers/stsb", split="test")
logging.info(stsb_eval_dataset)

logging.info("Load the Wikipedia dataset")
# Load the Wikipedia dataset: https://huggingface.co/datasets/sentence-transformers/wikipedia-en-sentences
wikipedia_train_dataset = load_dataset("sentence-transformers/wikipedia-en-sentences", split="train")
# Take 5000 random sentences from the Wikipedia dataset for evaluation
wikipedia_train_dataset_dict = wikipedia_train_dataset.train_test_split(test_size=5000)
wikipedia_train_dataset = wikipedia_train_dataset_dict["train"]
wikipedia_eval_dataset = wikipedia_train_dataset_dict["test"]
logging.info(wikipedia_train_dataset)

In [ ]:
# Concatenate the NLI and Wikipedia datasets for training
train_dataset: Dataset = concatenate_datasets([nli_train_dataset, wikipedia_train_dataset])

# Create a relatively small dataset for evaluation
eval_dataset: Dataset = concatenate_datasets([nli_eval_dataset.select(range(5000)), wikipedia_eval_dataset.select(range(5000))])

In [ ]:
len(train_dataset)

In [ ]:
len(eval_dataset)

In [ ]:
# Use the teacher model to get the gold embeddings
def map_embeddings(batch):
    return {
        "label": teacher_model.encode(batch["sentence"], show_progress_bar=False).tolist()
    }

In [ ]:
# Eval Dataset
eval_dataset = eval_dataset.map(map_embeddings)
eval_dataset.save_to_disk("datasets/distillation_evaluation_dataset_stsb-roberta")

In [ ]:
# Train Dataset
train_dataset = train_dataset.select(range(0, 4507105)).map(map_embeddings)
train_dataset.save_to_disk("datasets/distillation_train_dataset_stsb-roberta")